### Setup

In [ ]:
import csv
import math
import time
from dataclasses import dataclass
from pathlib import Path
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

In [ ]:
@dataclass(frozen=True)
class ShapeSpec:
    b: int
    c: int
    h: int
    w: int

SIZES         = (4, 8, 16, 32, 64, 128)
SHAPES        = [ShapeSpec(b=b, c=c, h=hw, w=hw) for b in SIZES for c in SIZES for hw in SIZES]
DEVICE        = "cuda"
DTYPE         = torch.float16
KERNEL_RADIUS = 2
WARMUP        = 1
RUNS          = 1
MODE          = "mean"
OUTPUT_LOG    = Path(f"redundancy_speed_{MODE}.csv") 

torch.set_printoptions(sci_mode=False)
torch.manual_seed(41)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(41)

print(f"[device]: {DEVICE}")

#### Max's Function

In [ ]:
def max_fn(feature_map: torch.Tensor, kernel_radius: int, mode: str = "max") -> torch.Tensor:
    if mode not in {"mean", "max"}:
        raise ValueError(f"mode must be 'mean' or 'max', got {mode!r}")

    eps        = 1e-6
    B, C, H, W = feature_map.shape  # [B, C, H, W]
    R          = int(kernel_radius)
    FM         = feature_map

    shifts: list[tuple[int, int]] = []
    for dh in range(-R, R + 1):
        for dw in range(-R, R + 1):
            if dh == 0 and dw == 0:
                continue
            shifts.append((dh, dw))

    corr       = torch.zeros((C, C), device=FM.device, dtype=FM.dtype)  # [C, C]
    corr_count = None
    
    if mode == "mean":
        corr_count = torch.zeros((C, C), device=FM.device, dtype=torch.int32)  # [C, C]

    for dh, dw in shifts:
        mh = H - abs(dh)
        mw = W - abs(dw)
        if mh <= 0 or mw <= 0:
            continue

        h1_start = max(0, -dh)
        h2_start = max(0, dh)
        w1_start = max(0, -dw)
        w2_start = max(0, dw)

        for c1 in range(C):
            for c2 in range(c1, C):
                m1 = FM[:, c1, h1_start : h1_start + mh, w1_start : w1_start + mw]  # [B, mh, mw]
                m2 = FM[:, c2, h2_start : h2_start + mh, w2_start : w2_start + mw]  # [B, mh, mw]

                m1_flat = m1.flatten()  # [B*mh*mw]
                m2_flat = m2.flatten()  # [B*mh*mw]

                m1_flat = (m1_flat - m1_flat.mean()) / m1_flat.std(unbiased=False).clamp_min(eps)  # [B*mh*mw]
                m2_flat = (m2_flat - m2_flat.mean()) / m2_flat.std(unbiased=False).clamp_min(eps)  # [B*mh*mw]

                r = torch.einsum("n,n->", m1_flat, m2_flat) / max(1, (B * mh * mw))
                r2 = r * r

                if mode == "max":
                    corr[c1, c2] = torch.maximum(corr[c1, c2], r2)
                else:
                    corr[c1, c2] += r2
                    corr_count[c1, c2] += 1

    mask = torch.triu(torch.ones((C, C), dtype=torch.bool, device=FM.device), diagonal=1)  # [C, C]

    if mode == "mean":
        corr = corr / corr_count.clamp_min(1).to(dtype=corr.dtype)  # [C, C]
        vals = corr.masked_select(mask)
        if vals.numel() == 0:
            return corr.new_zeros(())
        return vals.mean()

    corr = corr * mask
    return torch.amax(corr, dim=(0, 1))


#### Luca's Function

In [ ]:
def luca_fn(feature_map: torch.Tensor, kernel_radius: int, mode: str = "max") -> torch.Tensor:
    if mode not in {"mean", "max"}:
        raise ValueError(f"mode must be 'mean' or 'max', got {mode!r}")

    eps = 1e-6
    B, C, H, W = feature_map.shape  # [B, C, H, W]
    FM         = feature_map
    R          = int(kernel_radius)

    patch_HW = (2 * R) + 1
    patch_len = patch_HW * patch_HW

    patches = F.unfold(FM, kernel_size=patch_HW, padding=R) # [B, C*patch_len,  H*W]
    patches = patches.view(B, C, patch_len, H * W)          # [B, C, patch_len, H*W]

    center_k = patch_len // 2
    center   = patches[:, :, center_k, :]  # [B, C, H*W]

    center = center - center.mean(dim=(0, 2), keepdim=True)                                  # [B, C, H*W]
    center = center / center.std(dim=(0, 2), unbiased=False, keepdim=True).clamp_min(eps)     # [B, C, H*W]
    patches = patches - patches.mean(dim=(0, 3), keepdim=True)                               # [B, C, patch_len, H*W]
    patches = patches / patches.std(dim=(0, 3), unbiased=False, keepdim=True).clamp_min(eps)  # [B, C, patch_len, H*W]

    corr = torch.einsum("bip,bjkp->ijk", center, patches) / max(1, (B * int(center.shape[-1]))) # [C1, C2, K]
    corr = corr * corr                                                                           # [C1, C2, K]
    corr = torch.cat([corr[:, :, :center_k], corr[:, :, (center_k + 1):]], dim=-1)               # [C1, C2, K-1]

    if mode == "max":
        corr = corr.amax(dim=-1)  # [C, C]
    else:
        corr = corr.mean(dim=-1)  # [C, C]

    mask = torch.triu(torch.ones((C, C), dtype=torch.bool, device=FM.device), diagonal=1)
    if mode == "mean":
        vals = corr.masked_select(mask)
        if vals.numel() == 0:
            return corr.new_zeros(())
        return vals.mean()
    else:
        corr = corr * mask
        return torch.amax(corr, dim=(0, 1))

    

### Experiments

In [ ]:
def time_fn(
    fn,
    fm: torch.Tensor,
    kernel_radius: int,
    mode: str,
    warmup: int = 2,
    runs: int = 10,
) -> tuple[float, float]:
    # Warm up CUDA
    for _ in range(warmup):
        out = fn(fm, kernel_radius, mode)
    torch.cuda.synchronize()

    times: list[float] = []
    for _ in range(runs):
        t0 = time.perf_counter()
        out = fn(fm, kernel_radius, mode)
        torch.cuda.synchronize()
        t1 = time.perf_counter()
        times.append(t1 - t0)

    value = float(out.detach().float().cpu())
    return float(sum(times) / max(1, len(times))), value

In [ ]:
def make_fm(shape: ShapeSpec) -> torch.Tensor:
    return torch.randn(
        (shape.b, shape.c, shape.h, shape.w),
        device=DEVICE,
        dtype=DTYPE,
    )

In [ ]:
def benchmark(
    shapes: list[ShapeSpec],
    kernel_radius: int = 2,
    mode: str = "max",
    warmup: int = 2,
    runs: int = 10,
    output_log: Path | None = Path("redundancy_speed_outputs.csv"),
) -> list[dict]:
    fns = {
        "max_fn": max_fn
    }

    fieldnames = [
        "kernel_radius",
        "b",
        "c",
        "h",
        "w",
        "max_fn_sec",
        "max_fn_value",
        "luca_fn_sec",
        "luca_fn_value",
    ]

    if output_log is not None:
        output_log.parent.mkdir(parents=True, exist_ok=True)
        with output_log.open("w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()

    results: list[dict] = []
    for shape in tqdm(shapes):
        fm = make_fm(shape)
        row: dict = {
            "b": shape.b,
            "c": shape.c,
            "h": shape.h,
            "w": shape.w,
        }
        for name, fn in fns.items():
            sec, val = time_fn(fn, fm, kernel_radius=kernel_radius, mode=mode, warmup=warmup, runs=runs)
            row[f"{name}_sec"] = sec
            row[f"{name}_value"] = val


        if output_log is not None:
            with output_log.open("a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writerow({"kernel_radius": kernel_radius, **row})

        results.append(row)

    return results

In [ ]:
results = benchmark(
    SHAPES,
    kernel_radius=KERNEL_RADIUS,
    mode=MODE,
    warmup=WARMUP,
    runs=RUNS,
    output_log=OUTPUT_LOG,
)

print(f"[values] wrote {len(results)} rows to {OUTPUT_LOG.resolve()}")